# Create Work Summary Links (oxjob #2535)

One row per candidate summary link: `origin_work_id` (the summary), `target_work_id` (the work it summarizes), where
the link came from, and `served` (it passed every rule). `notebooks/end2end/CreateWorkLinks` unions the served rows into
`openalex.works.work_links` as link type `summary`; `/links` serves them.

Decided 22 Nov 2026 (oxjob #2535): a summary of another work (an abstracting service's abstract, a digest, a
research highlight, a Year Book entry, a clinical answer, a lay summary; agent-written summaries later) is its own work,
linked to the work it summarizes, credited to whoever wrote it; never merged with it.

**Services** (`openalex.works.summary_services`, written by the first cell from the list below): sources whose items
each summarize one work published elsewhere, found by the share of items that cite exactly one work on another source
within two years of it and read by hand (oxjob #2535 phase 1). Adding a service is one row here.

**Rules** (the first that fails names the drop): the summary cites exactly one work; that work is live (after merges),
on another source, and not the summary itself; the same service doesn't cite it from more than two summaries (a
congress abstract book that holds the summarized abstract, not the abstract itself); its title doesn't name a meeting,
congress, symposium, proceedings or abstracts book; the summary is dated from a year before to three years after it
unless the titles match; and the link is confirmed: the summary's own Crossref record cites the work's DOI (publisher
deposit, or Crossref's match of the publisher's reference string), or the titles match ('ChemInform Abstract: <title>').
Test build 22 Nov 2026: 972,574 links; fresh random 100 read by hand 100/100 right, top-up 50 from the smaller services
49/50.

Revert: git revert the oxjob #2535 commit; drop the work_summary_links* tables (the next Work Links run without the
union drops the rows from work_links and the links index).

In [ ]:
-- 1. The services: sources whose items each summarize one work published elsewhere (oxjob #2535 phase 1).
--    kind: abstracting-service, digest, research-highlights, year-book, clinical-answers, journal-club, lay-summary.
--    byline_rule (for the authors step, oxjob #2535): replace-with-organization = the records copy the summarized
--    paper's authors, so the byline becomes the organization that wrote the summary; organization-if-empty = the
--    service names no person, so the organization is the author; keep = the byline already names the summary's writer.
--    organization: the name the authors step credits (CreateWorksBase summary_org_bylines), only once its profile is
--    registered in openalex.authors.organization_author_profiles; NULL = not credited yet. 27 Nov 2026: the four Adis /
--    Springer Healthcare digests, whose items name no one (Crossref records and pages, random 100 read).
CREATE OR REPLACE TABLE openalex.works.summary_services
COMMENT 'oxjob #2535: sources whose items each summarize one work published elsewhere; read by CreateWorkSummaryLinks'
AS SELECT * FROM VALUES
  ('S41354064', 'abstracting-service', 'replace-with-organization', 'ChemInform', 'ChemInform'),
  ('S4393914274', 'abstracting-service', 'replace-with-organization', 'ChemInform', 'Chemischer Informationsdienst Organische Chemie (ChemInform''s first title)'),
  ('S14436165', 'digest', 'organization-if-empty', 'Reactions Weekly', 'Reactions Weekly'),
  ('S2764501393', 'digest', 'keep', NULL, 'Journal watch'),
  ('S135102583', 'research-highlights', 'keep', NULL, 'Synfacts'),
  ('S191882557', 'digest', 'organization-if-empty', 'PharmacoEconomics & Outcomes News', 'PharmacoEconomics & Outcomes News'),
  ('S11454955', 'digest', 'organization-if-empty', 'Inpharma Weekly', 'Inpharma Weekly'),
  ('S4210232662', 'clinical-answers', 'keep', NULL, 'Cochrane Clinical Answers'),
  ('S21668869', 'digest', 'organization-if-empty', NULL, 'Strategic Direction'),
  ('S4210192124', 'digest', 'organization-if-empty', NULL, 'Im Focus Onkologie'),
  ('S2739039196', 'digest', 'keep', NULL, 'Pediatric Neurology Briefs'),
  ('S126990314', 'digest', 'keep', NULL, 'Medisch-Farmaceutische Mededelingen'),
  ('S118223562', 'digest', 'organization-if-empty', NULL, 'Advances in Small Animal Medicine and Surgery'),
  ('S4210174073', 'digest', 'organization-if-empty', NULL, 'Neonatologie Scan'),
  ('S4210169559', 'digest', 'organization-if-empty', NULL, 'Handchirurgie Scan'),
  ('S4210171787', 'digest', 'organization-if-empty', NULL, 'Gefaessmedizin Scan'),
  ('S4210188265', 'digest', 'organization-if-empty', NULL, 'Neuroradiologie Scan'),
  ('S65923806', 'digest', 'organization-if-empty', NULL, 'TumorDiagnostik & Therapie'),
  ('S4210172675', 'digest', 'organization-if-empty', NULL, 'Onkologische Welt'),
  ('S4210191460', 'digest', 'organization-if-empty', NULL, 'Aktuelle Kardiologie'),
  ('S4210186150', 'journal-club', 'organization-if-empty', NULL, 'Journal Club AINS'),
  ('S4210217833', 'digest', 'organization-if-empty', 'Springer Healthcare News', 'Springer Healthcare News'),
  ('S69799158', 'digest', 'organization-if-empty', NULL, 'Human Resource Management International Digest'),
  ('S196593774', 'digest', 'organization-if-empty', NULL, 'Development in Learning Organizations'),
  ('S4210232289', 'digest', 'organization-if-empty', NULL, 'Annals in Social Responsibility'),
  ('S7407090035', 'digest', 'keep', NULL, 'Spotlight by CeFEO'),
  ('S4210211574', 'digest', 'organization-if-empty', NULL, 'Lippincott''s Bone and Joint Newsletter'),
  ('S4210197562', 'digest', 'organization-if-empty', NULL, 'MSK Muskuloskelettale Physiotherapie'),
  ('S4220651502', 'digest', 'organization-if-empty', NULL, 'Angewandte Nuklearmedizin'),
  ('S4210215159', 'digest', 'organization-if-empty', NULL, 'Allgemeinmedizin up2date'),
  ('S182558529', 'digest', 'organization-if-empty', NULL, 'Perspectives on Sexual and Reproductive Health (Digests)'),
  ('S4306529286', 'lay-summary', 'keep', NULL, 'Science Trends (authors'' own lay summaries)'),
  ('S4387278989', 'research-highlights', 'keep', NULL, 'Autophagy Reports (authors'' own highlights)'),
  ('S4210235921', 'research-highlights', 'organization-if-empty', NULL, 'BoneKEy Reports'),
  ('S4210183954', 'research-highlights', 'organization-if-empty', NULL, 'Radiology Imaging Cancer'),
  ('S93373720', 'research-highlights', 'keep', NULL, 'Nature Reviews Immunology'),
  ('S26843219', 'research-highlights', 'keep', NULL, 'Nature Reviews Neuroscience'),
  ('S56802129', 'research-highlights', 'keep', NULL, 'Nature Reviews Microbiology'),
  ('S18618073', 'research-highlights', 'keep', NULL, 'Nature Reviews Nephrology'),
  ('S67216020', 'research-highlights', 'keep', NULL, 'Nature Reviews Urology'),
  ('S160464432', 'research-highlights', 'keep', NULL, 'Nature Reviews Cancer'),
  ('S78310130', 'research-highlights', 'keep', NULL, 'Nature Reviews Rheumatology'),
  ('S116269268', 'research-highlights', 'keep', NULL, 'Nature Reviews Endocrinology'),
  ('S120397320', 'research-highlights', 'keep', NULL, 'Nature Reviews Gastroenterology & Hepatology'),
  ('S8553189', 'research-highlights', 'keep', NULL, 'Nature Reviews Genetics'),
  ('S41981144', 'research-highlights', 'keep', NULL, 'Nature Reviews Clinical Oncology'),
  ('S62674001', 'research-highlights', 'keep', NULL, 'Nature Reviews Cardiology'),
  ('S22232893', 'research-highlights', 'keep', NULL, 'Nature Reviews Neurology'),
  ('S196724125', 'research-highlights', 'keep', NULL, 'Nature Reviews Molecular Cell Biology'),
  ('S2764917516', 'research-highlights', 'keep', NULL, 'Nature Reviews Materials'),
  ('S4210238410', 'research-highlights', 'keep', NULL, 'Nature India'),
  ('S4210237454', 'research-highlights', 'keep', NULL, 'Nature China'),
  ('S4210223298', 'research-highlights', 'keep', NULL, 'Nature Italy'),
  ('S46371784', 'research-highlights', 'keep', NULL, 'NPG Asia Materials'),
  ('S4210200363', 'research-highlights', 'keep', NULL, 'Nature Reports Stem Cells'),
  ('S75976234', 'research-highlights', 'keep', NULL, 'Nature Clinical Practice Gastroenterology & Hepatology'),
  ('S58875525', 'research-highlights', 'keep', NULL, 'Nature Clinical Practice Urology'),
  ('S4210176377', 'research-highlights', 'keep', NULL, 'Nature Clinical Practice Oncology'),
  ('S4210170635', 'research-highlights', 'keep', NULL, 'Nature Clinical Practice Cardiovascular Medicine'),
  ('S51708466', 'research-highlights', 'keep', NULL, 'Nature Clinical Practice Endocrinology & Metabolism'),
  ('S4210180506', 'research-highlights', 'keep', NULL, 'Nature Clinical Practice Rheumatology'),
  ('S4210172272', 'research-highlights', 'keep', NULL, 'Nature Clinical Practice Neurology'),
  ('S4210168417', 'year-book', 'keep', NULL, 'Year Book of Psychiatry and Applied Mental Health'),
  ('S4210170199', 'year-book', 'keep', NULL, 'Year Book of Surgery'),
  ('S30421737', 'year-book', 'keep', NULL, 'Year Book of Dermatology and Dermatologic Surgery'),
  ('S54421717', 'year-book', 'keep', NULL, 'Year Book of Orthopedics'),
  ('S4210240726', 'year-book', 'keep', NULL, 'Year Book of Obstetrics, Gynecology and Women''s Health'),
  ('S4210197334', 'year-book', 'keep', NULL, 'Year Book of Anesthesiology and Pain Management'),
  ('S4210222886', 'year-book', 'keep', NULL, 'Year Book of Plastic and Aesthetic Surgery'),
  ('S47614163', 'year-book', 'keep', NULL, 'Year Book of Urology'),
  ('S106929689', 'year-book', 'keep', NULL, 'Year Book of Vascular Surgery'),
  ('S153802034', 'year-book', 'keep', NULL, 'Year Book of Pathology and Laboratory Medicine'),
  ('S105178564', 'year-book', 'keep', NULL, 'Year Book of Otolaryngology-Head and Neck Surgery'),
  ('S4210223995', 'year-book', 'keep', NULL, 'Year Book of Oncology'),
  ('S4210171824', 'year-book', 'keep', NULL, 'Year Book of Neurology and Neurosurgery'),
  ('S4210167900', 'year-book', 'keep', NULL, 'Year Book of Ophthalmology'),
  ('S191942090', 'year-book', 'keep', NULL, 'Year Book of Hand and Upper Limb Surgery'),
  ('S4210193464', 'year-book', 'keep', NULL, 'Year Book of Gastroenterology'),
  ('S4210180476', 'year-book', 'keep', NULL, 'Year Book of Medicine'),
  ('S4210188930', 'year-book', 'keep', NULL, 'Year Book of Diagnostic Radiology'),
  ('S139637901', 'year-book', 'keep', NULL, 'Year Book of Sports Medicine'),
  ('S50342286', 'year-book', 'keep', NULL, 'Year Book of Pulmonary Disease'),
  ('S4210239590', 'year-book', 'keep', NULL, 'Year Book of Cardiology'),
  ('S202586344', 'year-book', 'keep', NULL, 'Year Book of Critical Care Medicine'),
  ('S4210226369', 'year-book', 'keep', NULL, 'Year Book of Pediatrics'),
  ('S65098588', 'year-book', 'keep', NULL, 'Year Book of Endocrinology'),
  ('S4210202356', 'year-book', 'keep', NULL, 'Year Book of Neonatal and Perinatal Medicine'),
  ('S136007311', 'year-book', 'keep', NULL, 'Year Book of Dentistry'),
  ('S2764352954', 'year-book', 'keep', NULL, 'Breast Diseases: A Year Book Quarterly')
AS t(source_id, kind, byline_rule, organization, name)

In [ ]:
-- 2. Candidates: every item on a service that cites exactly one work, with the evidence each rule reads.
CREATE OR REPLACE TABLE openalex.works.work_summary_links_candidates AS
WITH svc AS (SELECT concat('https://openalex.org/', source_id) AS sid, source_id, kind FROM openalex.works.summary_services),
s AS (
  SELECT w.id AS origin_work_id, svc.source_id, svc.kind, w.title, w.publication_year AS yr,
         regexp_replace(lower(w.doi), '^https?://(dx[.])?doi[.]org/', '') AS sdoi,
         CAST(get(w.referenced_works, 0) AS BIGINT) AS ref_id
  FROM openalex.works.openalex_works w JOIN svc ON w.primary_location.source.id = svc.sid
  WHERE size(w.referenced_works) = 1
),
r AS (
  SELECT s.*, coalesce(m.merge_into_id, s.ref_id) AS target_work_id
  FROM s LEFT JOIN openalex.works.merged_work_ids_map m ON m.id = s.ref_id
),
o AS (
  SELECT id, title, publication_year AS oyr, primary_location.source.id AS osid,
         regexp_replace(lower(doi), '^https?://(dx[.])?doi[.]org/', '') AS odoi
  FROM openalex.works.openalex_works WHERE id IN (SELECT target_work_id FROM r)
),
cr AS (
  SELECT native_id AS sdoi,
         transform(filter(reference, x -> x.DOI IS NOT NULL), x -> regexp_replace(lower(trim(x.DOI)), '^(https?://(dx[.])?doi[.]org/|doi:)', '')) AS ref_dois
  FROM openalex.crossref.crossref_deduplicated WHERE native_id IN (SELECT sdoi FROM s WHERE sdoi IS NOT NULL)
),
j AS (
  SELECT r.origin_work_id, r.target_work_id, r.source_id, r.kind, r.yr, o.oyr, o.id AS target_live,
         o.osid = concat('https://openalex.org/', r.source_id) AS same_source,
         coalesce(array_contains(cr.ref_dois, o.odoi), false) AS crossref_cites_target,
         coalesce(length(regexp_replace(o.title, '[^A-Za-z0-9]', '')) >= 20
                  AND endswith(lower(regexp_replace(r.title, '[^A-Za-z0-9]', '')), lower(regexp_replace(o.title, '[^A-Za-z0-9]', ''))),
                  false) AS title_match,
         o.title RLIKE '(?i)(congress|annual meeting|abstract book|abstracts|symposium|conference|proceedings|meeting)' AS container_title
  FROM r LEFT JOIN o ON o.id = r.target_work_id LEFT JOIN cr ON cr.sdoi = r.sdoi
)
SELECT j.*, count(*) OVER (PARTITION BY j.source_id, j.target_work_id) AS same_service_fanin,
  CASE
    WHEN j.target_live IS NULL THEN 'summarized work not live'
    WHEN j.origin_work_id = j.target_work_id THEN 'same work'
    WHEN j.same_source THEN 'same source'
    WHEN count(*) OVER (PARTITION BY j.source_id, j.target_work_id) > 2 THEN 'container (the service cites it from 3+ summaries)'
    WHEN j.container_title THEN 'container (meeting or abstracts book title)'
    WHEN NOT (j.yr BETWEEN j.oyr - 1 AND j.oyr + 3) AND NOT j.title_match THEN 'out of window'
    WHEN NOT (j.crossref_cites_target OR j.title_match) THEN 'not confirmed'
  END AS drop_reason
FROM j

In [ ]:
-- 3. One row per candidate; `served` marks the rows that passed every rule (a summary cites one work, so each edge
--    has one row). Built beside the served table, checked, then swapped in.
CREATE OR REPLACE TABLE openalex.works.work_summary_links_build AS
SELECT origin_work_id, target_work_id, source_id AS service_source_id, kind,
       CASE WHEN crossref_cites_target THEN 'crossref-reference' ELSE 'title-match' END AS link_source,
       drop_reason IS NULL AND target_work_id IS NOT NULL AS served, drop_reason
FROM openalex.works.work_summary_links_candidates

In [ ]:
-- 4. Fuse: refuse a build that serves 20% fewer links than the one being replaced (a broken input, not real change).
CREATE TABLE IF NOT EXISTS openalex.works.work_summary_links LIKE openalex.works.work_summary_links_build

In [ ]:
SELECT assert_true(
  (SELECT count_if(served) FROM openalex.works.work_summary_links_build)
    >= 0.8 * (SELECT count_if(served) FROM openalex.works.work_summary_links),
  'work_summary_links_build serves over 20% fewer summary links than work_summary_links: check the inputs before swapping'
) AS fuse_ok

In [ ]:
-- 5. Swap in.
CREATE OR REPLACE TABLE openalex.works.work_summary_links
COMMENT 'oxjob #2535: summary links (origin = the summary, target = the work it summarizes); served rows go to work_links as summary'
AS SELECT * FROM openalex.works.work_summary_links_build

In [ ]:
-- 6. Report: served links by kind and the drops by reason (display only).
SELECT kind, count(*) AS candidates, count_if(served) AS served, count_if(drop_reason = 'not confirmed') AS not_confirmed,
       count_if(drop_reason LIKE 'container%') AS containers, count_if(drop_reason = 'out of window') AS out_of_window
FROM openalex.works.work_summary_links GROUP BY ROLLUP(kind) ORDER BY served DESC